# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/Users/vkalashnikov/Documents/repos/ai-maker-space/AIEC1/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.16
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/var/folders/z8/pbvjy1gs1d3b_rd53b4knls80000gp/T/ipykernel_53550/1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [6]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [7]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [9]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:38<00:00,  1.91s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [01:02<00:00,  1.05s/it]
Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 173.34it/s]

KnowledgeGraph(nodes: 20, relationships: 57)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [10]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 57
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [11]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts/cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 57)


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer

**What the transforms added.** The raw input was 20 LangChain documents, each holding only `page_content` and PDF metadata. After running `default_transforms_for_prechunked` minus `CustomNodeFilter`, every node also has:

- **`summary`** — an LLM-written compression of the page. This becomes the unit the multi-hop synthesizers reason over, because comparing summaries is cheaper and more semantically meaningful than comparing raw text.
- **`summary_embedding`** — a dense vector for the summary, computed via `OpenAIEmbeddings` through AI Gateway. Used to score node-to-node similarity.
- **`themes`** — short topical labels (e.g., "preventive care," "feeding behavior"). These power thematic clustering in the abstract multi-hop synthesizer.
- **`entities`** — named entities extracted from the page (life stages, conditions, body systems). These power overlap-based linking in the specific multi-hop synthesizer.

The transform pipeline output confirms this — every node now carries `['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']` instead of the original two fields.

**Why the two relationship types matter.** The relationship builders are `CosineSimilarityBuilder` (over `summary_embedding`) and `OverlapScoreBuilder` (over `entities`). They build two complementary edge sets:

- **`cosine_similarity` edges** — "these two pages are *about* the same thing in a fuzzy, semantic sense." This is what `MultiHopAbstractQuerySynthesizer` uses to find pairs of pages that share themes without sharing exact phrasing. Without it, abstract multi-hop questions ("connect these two broad concepts") have no traversal scaffold to walk.
- **`overlap_score` edges** — "these two pages share concrete named entities." This is what `MultiHopSpecificQuerySynthesizer` uses to find pairs where a specific factual answer requires combining details from both pages (e.g., the senior-life-stage page and the feeding-amount page both reference the same age range). Without it, specific multi-hop questions degenerate into single-hop questions.

The two builders model two different forms of "relatedness," and the two multi-hop synthesizers depend on different forms. That's why dropping either builder would collapse a query type — the default helper at Task 4 only lists the synthesizers the *enriched* graph can actually support. With both builder types present, all three synthesizers are available and the 57 relationships across 20 nodes give the generator enough connectivity to construct genuine multi-hop scenarios.

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [12]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [13]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

**The three query types.**

- **Single-hop specific.** "What does the corpus say about X?" — one concrete fact lives in one chunk, and the right answer is essentially a quote-or-paraphrase from that chunk. Tests whether the retriever returns the right page and whether the answer model can copy out the relevant sentence without confabulating. Example from our dataset: *"Why is a life stage assessment important for cats?"*
- **Multi-hop specific.** "Given X and Y, what should Z be?" — the answer requires combining two or more *concrete* details that live in different chunks. There is still a single, verifiable factual answer, but no single chunk contains it; the retriever must surface both source chunks and the answer model must compose them. Example: *"For mature adult cats aged 7 to 10 years, how should daily feeding amounts be determined?"* (combines body-condition score, daily-energy calculation, and weight-monitoring guidance from separate pages).
- **Multi-hop abstract.** "How does theme A relate to theme B?" — the answer is a synthesis of concepts that may be implicit across multiple chunks. No single chunk contains the synthesis statement; the answer model has to reason across the retrieved set rather than extract. Example: *"For a cat with 'feeding behavior and environmental...'"* type questions that connect feeding behavior to environmental enrichment.

**Hardest for a basic dense-retrieval RAG: multi-hop specific.** Three compounding reasons:

1. **Retrieval is the bottleneck, and dense retrieval is single-pass.** A vanilla `vector_store.as_retriever(k=3)` ranks by similarity to the *question* embedding. Multi-hop specific questions encode both facets of the question into one query vector, which pulls in chunks similar to *the question as a whole* rather than chunks each containing one of the two needed pieces. Single-hop questions don't have this problem because their question embedding is close to the one chunk that answers them. Multi-hop abstract questions are forgiving because *any* on-theme chunk contributes; multi-hop specific questions are unforgiving because a *specific* sub-chunk is required.
2. **The answer model is graded on a strict reference.** For abstract questions, the reference answer is a synthesis and the judge tolerates paraphrase. For specific multi-hop, the reference contains concrete facts (numbers, age ranges, named procedures) and the correctness judge will mark down any missing sub-claim. Miss one of the two source chunks at retrieval time and the answer is provably incomplete.
3. **Failure modes stack.** If the retriever misses *either* required chunk, the answer model has no way to recover — it cannot extract a fact that isn't in front of it. Single-hop only has to get one thing right; multi-hop specific has to get N things right *together*.

Counter-evidence to consider: multi-hop *abstract* can also be hard because the judge has to score a more diffuse output. But its retrieval needs are looser (any thematically related chunk helps), so a basic dense retriever usually does well enough on recall to give the answer model something to synthesize from. The hardest combination for a basic dense-retrieval RAG is therefore *specific* questions that require *multiple* chunks — which is exactly what the Task 10 k=3 → k=6 experiment is targeting.

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [14]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:12<00:00,  2.12s/it]


,user_input,reference,synthesizer_name
0,How does the 2021 AAHA/AAFP Feline Life Stage ...,Jodi Westropp is listed among the authors of t...,single_hop_specific_query_synthesizer
1,Why is a life stage assessment important for c...,A life stage assessment is important because a...,single_hop_specific_query_synthesizer
2,"For a cat with ""feeding behavior and environme...",The context says cats naturally prefer to eat ...,multi_hop_abstract_query_synthesizer
3,"According to the feline life stage guidelines,...",Detecting signs of pain or anxiety and evaluat...,multi_hop_abstract_query_synthesizer
4,"According to JAAHA.ORG, how should i connect t...",JAAHA.ORG says that in mature adult and senior...,multi_hop_specific_query_synthesizer
5,"For Senior Cats, how should daily feeding amou...","For mature adult cats aged 7 to 10 years, dail...",multi_hop_specific_query_synthesizer


In [15]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts/cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

**Unrolled path (what this notebook does).** Build the knowledge graph explicitly, apply transforms one at a time, inspect node properties and relationship counts, save the graph to `artifacts/cat_health_knowledge_graph.json`, then call `TestsetGenerator(knowledge_graph=...).generate(...)`.

- **Pros:**
  - **Inspectable at every stage.** Cell 20's diagnostic loop confirms that nodes have `summary`, `summary_embedding`, `themes`, `entities` *before* generation starts. If a transform silently produced empty summaries, the `RuntimeError` guard in the per-transform loop catches it instead of letting Ragas waste API calls on a broken graph.
  - **Reusable artifact.** The saved graph survives kernel restarts. Tweaking the query distribution or the test-set size doesn't re-run the expensive transforms — those LLM calls happen once.
  - **Custom-transform-friendly.** Filtering out `CustomNodeFilter` (which assumes a parent-child node structure we don't have) and swapping in `NonEmptySummary` as the summary `output_model` are both natural here; the one-call helper hides those seams.
  - **Better failure attribution.** A failure during `EmbeddingExtractor` is obviously the embeddings model; a failure during `CosineSimilarityBuilder` is obviously the relationship math. Mixed-up traces are harder to diagnose.

- **Cons:**
  - **More code to maintain.** Three explicit functions (`build_prechunked_knowledge_graph`, `apply_transforms`, `TestsetGenerator.generate`) versus one.
  - **Easier to drift from upstream defaults.** If Ragas updates `default_transforms_for_prechunked` to add a new builder, the unrolled path needs to be re-read; the one-call helper picks it up for free.

**One-call path (`generate_with_chunks`).** A single helper that takes raw chunks, builds the graph internally, runs default transforms, and emits a test set.

- **Pros:**
  - **Cheapest to write and read.** Three or four lines vs ~80.
  - **Tracks Ragas upstream defaults automatically.** No risk of forgetting a new transform or new relationship builder.
  - **Right shape for prototyping** when you don't yet care about the internals.

- **Cons:**
  - **Opaque on failure.** If summaries come out blank or the relationship count is suspiciously low, you have to bisect after the fact.
  - **No graph artifact.** Every re-run repeats the full enrichment cost. The bill is not trivial — in this notebook, transform application took ~100s and dozens of LLM calls.
  - **Hard to customize.** Swapping a transform output model or filtering one transform out of the pipeline requires reaching past the helper, which negates its convenience.

**When to choose each.**

- **Choose the unrolled path** when you (a) intend to iterate on the dataset, the distribution, or the synthesizers; (b) need to debug bad generations; (c) want graph reuse across notebook runs or across notebooks; or (d) are running expensive transforms and want to fail fast on errors. That describes our session: teaching the *mechanics*, with iterative review in Activity #1, makes inspectability worth the extra code.
- **Choose the one-call path** when you are sketching out a first test set, the corpus is small, the cost is negligible, and you have no plans to revisit the graph. Also good for CI/automation where the graph is recomputed every run anyway.

A reasonable hybrid is to develop unrolled, lock in a transform pipeline that works for your corpus, then collapse to `generate_with_chunks` for repeated team use once the choices are stable.

## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [18]:
# Activity #1 workspace
#
# Reviewed the six generated examples against the criteria in the activity
# brief (answerable, grounded, natural, non-duplicate, safe).
#
# Findings:
# - Row 0: "How does the 2021 AAHA/AAFP Feline Life Stage..." resolves to
#   "Jodi Westropp is listed among the authors..." -- a trivia question about
#   paper authorship. Not useful for evaluating a cat health RAG, drop it.
# - Row 4: "According to JAAHA.ORG, how should i connect t..." has garbled
#   phrasing and an odd "JAAHA.ORG" framing that no real user would type.
#   Rewrite into a natural question and strip the JAAHA.ORG reference from the
#   reference answer so the judge does not penalize the RAG for omitting a
#   URL.
# - Row 5: question asks about "Senior Cats" but the reference describes
#   "mature adult cats aged 7 to 10 years." Realign the question to match the
#   life stage the reference actually supports.
# - Rows 1, 2, 3 are grounded and natural; keep as-is.

approved_testset_df = testset_df.drop(index=[0]).reset_index(drop=True)

# Row 4 in the original frame is index 3 after dropping row 0.
approved_testset_df.loc[3, "user_input"] = (
    "In mature adult and senior cats, how often should owners and the "
    "veterinary team connect to monitor for changes?"
)
approved_testset_df.loc[3, "reference"] = (
    "In mature adult and senior cats, owners and the veterinary team "
    "should stay in regular contact to detect changes early and identify "
    "concerns between scheduled visits."
)

# Row 5 in the original frame is index 4 after dropping row 0.
approved_testset_df.loc[4, "user_input"] = (
    "For mature adult cats aged 7 to 10 years, how should daily feeding "
    "amounts be determined?"
)

review_status = "student_reviewed"

print(f"Approved examples: {len(approved_testset_df)}")
print("Examples by synthesizer after review:")
print(approved_testset_df["synthesizer_name"].value_counts())

display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Approved examples: 5
Examples by synthesizer after review:
synthesizer_name
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
single_hop_specific_query_synthesizer    1
Name: count, dtype: int64


,user_input,reference_contexts,reference,synthesizer_name
0,Why is a life stage assessment important for c...,[Introduction\nThe feline patient ’s life stag...,A life stage assessment is important because a...,single_hop_specific_query_synthesizer
1,"For a cat with ""feeding behavior and environme...","[<1-hop>\n\n10 months, primarily by phone cont...",The context says cats naturally prefer to eat ...,multi_hop_abstract_query_synthesizer
2,"According to the feline life stage guidelines,...",[<1-hop>\n\nDetecting signs of pain or anxiety...,Detecting signs of pain or anxiety and evaluat...,multi_hop_abstract_query_synthesizer
3,"In mature adult and senior cats, how often sho...",[<1-hop>\n\ndetection of changes and identi ﬁc...,"In mature adult and senior cats, owners and th...",multi_hop_specific_query_synthesizer
4,"For mature adult cats aged 7 to 10 years, how ...",[<1-hop>\n\ngood starting point is to calculat...,"For mature adult cats aged 7 to 10 years, dail...",multi_hop_specific_query_synthesizer


### 📝 Activity #1 Notes

#### Goal

The Ragas test set generator emits *candidate* synthetic examples — questions, reference contexts pulled from the knowledge graph, and a generated reference answer. Per the activity brief, every row was checked against five criteria before becoming eval data:

1. **Answerable** from the provided reference contexts
2. **Grounded** — reference answer fully supported by those contexts
3. **Natural** phrasing a plausible user would actually type
4. **Non-duplicate**
5. **Safety-preserving** (no diagnosis / prescription leakage)

The deliverable is a curated `approved_testset_df` plus written justification, with `review_status = "student_reviewed"` so the gate in Task 6 allows upload to LangSmith.

#### Inputs reviewed

Six Ragas-generated rows from `testset_df`, distributed 2 / 2 / 2 across the three synthesizers:

| Idx | Synthesizer | Question (truncated) | Reference (truncated) |
|---|---|---|---|
| 0 | single_hop_specific | "How does the 2021 AAHA/AAFP Feline Life Stage..." | "Jodi Westropp is listed among the authors..." |
| 1 | single_hop_specific | "Why is a life stage assessment important for cats..." | "A life stage assessment is important because..." |
| 2 | multi_hop_abstract | "For a cat with 'feeding behavior and environmental...'" | "The context says cats naturally prefer to eat..." |
| 3 | multi_hop_abstract | "According to the feline life stage guidelines..." | "Detecting signs of pain or anxiety and evaluating..." |
| 4 | multi_hop_specific | "According to JAAHA.ORG, how should i connect t..." | "JAAHA.ORG says that in mature adult and senior..." |
| 5 | multi_hop_specific | "For Senior Cats, how should daily feeding amount..." | "For mature adult cats aged 7 to 10 years, daily feeding..." |

#### Per-row decisions

##### Row 0 — Removed

- **Issue:** Question reduces to paper-authorship trivia ("who wrote the guidelines"). Grounded in the PDF's title page but tests metadata retrieval, not cat health knowledge.
- **Why drop instead of edit:** The whole point of this single-hop example is the wrong target behavior for the application. A correct RAG would arguably refuse this question, which would *hurt* the correctness judge score. Keeping the row would inject noise into Task 10's k=3 vs k=6 comparison without measuring anything we care about.
- **Safety:** No safety issue, but misrepresents real usage.

##### Row 4 — Edited (question + reference)

- **Issue A — naturalness:** `"According to JAAHA.ORG, how should i connect t..."` is truncated and the `JAAHA.ORG` framing is unnatural. Real users ask about cats, not what a journal website says.
- **Issue B — groundedness leak:** Reference answer started with `"JAAHA.ORG says that..."`. A faithful RAG that summarizes the *content* without naming the URL would be penalized by the groundedness judge, even though it's correct.
- **Fix:**
  - Question rewritten to: *"In mature adult and senior cats, how often should owners and the veterinary team connect to monitor for changes?"*
  - Reference rewritten to drop the URL preamble and keep only the substantive claim about owner/vet communication and early change detection.
- **Safety:** Stayed within "monitor for changes / identify concerns" language — no diagnostic or prescriptive content introduced.

##### Row 5 — Edited (question only)

- **Issue:** Question/reference life-stage mismatch. Question asked about **Senior Cats**, but the supporting context and reference answer describe **mature adult cats aged 7 to 10 years**. AAHA defines those as different life stages, so a RAG returning the correct senior-cat guidance would look wrong against this reference.
- **Fix:** Aligned the question to the life stage the contexts actually support: *"For mature adult cats aged 7 to 10 years, how should daily feeding amounts be determined?"*
- **Why edit instead of drop:** The reference answer itself is solid and grounded. Only the question label was wrong, which is a cheap one-line fix that preserves multi-hop coverage.

##### Rows 1, 2, 3 — Kept unchanged

Each is answerable from its `reference_contexts`, naturally phrased, non-duplicative, and stays within the corpus's educational framing. No edits required.

#### Resulting dataset

- **6 generated → 5 approved** (1 removed, 2 edited, 3 kept).
- Distribution after review: 1 single-hop specific / 2 multi-hop abstract / 2 multi-hop specific — all three synthesizer types still represented, so the Task 10 k=3 vs k=6 comparison can still slice results by query type.
- `review_status` set to `"student_reviewed"`.

#### Effect on downstream tasks

- **Task 6 gate passed.** Cell 39's `ValueError` traceback is gone; LangSmith dataset `aim-session-5-cat-health-synthetic-5aabf2e8` was created with 5 examples.
- **Cleaner experiments downstream.** Removing row 0 eliminates a guaranteed false-negative (authorship trivia), and the row 5 realignment removes another (senior vs mature-adult mismatch). Both fixes mean the k=3 vs k=6 deltas in Tasks 9–10 reflect actual retrieval-quality differences rather than dataset defects.
- **Provenance retained.** `synthesizer_name`, `synthetic_reference: True`, and `review_status: "student_reviewed"` flow through `metadata` on each LangSmith example, so per-query-type breakdowns and audit trails remain available.

#### What was *not* done (and why)

- **No new examples added.** The activity asks to "remove or repair at least one weak example"; it does not require padding back to 6. Five reviewed examples is honest; synthesizing more without grounding them would dilute quality.
- **No `<N-hop>` marker stripping in `reference_contexts`.** A possible bias source for retrieval-relevance judging, but left untouched here because the activity is scoped to question/reference review, not context post-processing. Worth revisiting after seeing Task 10's judge scores.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [19]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-5aabf2e8
Examples uploaded: 5


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

Each of the three keys answers a different downstream question, and together they let LangSmith experiments be sliced, audited, and curated without round-tripping back to the generator.

- **`synthesizer_name` — slice-by-query-type analysis.** Aggregate scores like "average correctness = 0.78" hide the fact that single-hop and multi-hop questions stress completely different parts of the application. With this key on each example, the LangSmith compare view (or a quick `pandas.groupby` on the feedback export) reveals whether a configuration change like k=3 → k=6 helped *all* query types or only multi-hop specific ones. The Activity #2 third experiment is designed around exactly this slicing — the prediction is per-synthesizer-type, not a single mean. Discarding the key collapses three signals into one.

- **`synthetic_reference: True` — provenance for the eval boundary.** Synthetic references are *not* ground truth, even after Activity #1 review. Six months from now, someone will look at this dataset and need to know whether the references came from a human SME (defensible to ship answers against), a Ragas generator (defensible for relative comparison, *not* absolute), or a previous production deployment (yet another category). Without this flag, the dataset has no audit trail. With it, anyone who later mixes synthetic and human examples can filter the synthetic subset out for any decision that needs ground truth.

- **`review_status: "student_reviewed"` — separates curated from raw.** Reviewed examples passed five criteria (answerable, grounded, natural, non-duplicate, safety-preserving). Unreviewed ones did not. Keeping the status visible means later loaders can filter to `review_status == "student_reviewed"` for the "gold" split and treat unreviewed examples as a separate, larger, lower-quality corpus. It also gates uploads — Task 6 actually checks this value before allowing `create_examples` to run, which is a cheap way to prevent the "I forgot to do Activity #1" failure mode.

**Why metadata specifically, not inputs or outputs.** The judges in Task 8 consume only `inputs.question`, `outputs.answer`, and `outputs.reference_contexts`. Putting provenance into `inputs` or `outputs` would leak it to the LLM-as-judge prompts, which would bias scores (e.g., a judge that learns it's looking at synthetic data may grade more leniently). Metadata is the right place because LangSmith treats it as orthogonal to the eval payload: visible to humans, invisible to judges.

**Practical follow-up.** Two more keys would make this even more useful and cost nothing to add:

- `query_complexity: "single_hop" | "multi_hop"` derived from `synthesizer_name`, so slicing is easier without string parsing.
- `original_synthesizer_row_id` so you can trace back to the exact pre-curation Ragas output if you ever need to compare reviewed vs raw versions of the same row.

Both are downstream conveniences, not corrections to what's already there.

## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [20]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [21]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [22]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [23]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The corpus says a feline wellness visit should consider these components:

- Behavior and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes that wellness visits may include:
- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detailed breakdowns of each component.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat’s life stage. These recommenda

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [24]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [25]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

**Worked example: high correctness, low groundedness.**

Question (from our dataset): *"For mature adult cats aged 7 to 10 years, how should daily feeding amounts be determined?"*

Reference answer: *"For mature adult cats aged 7 to 10 years, daily feeding amounts should be calculated from the cat's body weight as a starting point and adjusted based on body condition score and weight monitoring over time."*

Imagine the RAG, at `k=3`, retrieves two chunks about life-stage care but *not* the chunk that contains the explicit body-weight calculation formula. The answer model nevertheless produces:

> "Daily feeding amounts for mature adult cats aged 7–10 years should be calculated from body weight as a starting point and refined using body-condition scoring with regular weight monitoring."

Two judge outcomes:

- **`answer_correctness` — high.** The judge compares the answer to the *reference*, sees the same key claims (body weight starting point + body-condition score + weight monitoring), and rates ~0.9.
- **`answer_groundedness` — low.** The judge compares the answer to the *retrieved contexts*, sees that none of them actually contain "body weight as a starting point" or "weight monitoring," and rates ~0.3.

The disagreement is the diagnostic signal: **the model knew the right answer from training data, not from the corpus.** That's a hallucination of *the right thing*, which is the worst kind because it passes any correctness check that only compares to a reference.

**What to inspect in the trace.**

1. **`outputs.contexts` first.** Confirm that the retrieved chunks really don't contain the claim. If they do contain it, the groundedness judge mis-graded — re-read its rationale and consider whether the corpus phrasing was too oblique for the judge to match. The fix is judge prompt or different model, not the application.
2. **Then the answer text.** Are there extractive markers (page numbers, exact phrasing from the corpus, "the guidelines state...")? If yes, the model thinks it's quoting the corpus and the retrieval *should* have included that page — diagnosis is retrieval miss, fix is higher `k`, better embeddings, or rerank. If no — the answer is paraphrased general veterinary knowledge — diagnosis is *prior-knowledge leak* and the fix is a stricter prompt ("answer only what the retrieved context says; otherwise say the corpus does not provide enough information").
3. **The system prompt.** `RAG_SYSTEM_PROMPT` already says "Answer the question using only the retrieved context. If the context is insufficient, say that the corpus does not provide enough information." A high-correctness-low-groundedness pair means the model ignored that instruction. Either tighten the prompt (add an explicit "do not draw on outside knowledge") or downgrade to a smaller/more-instruction-following model and re-evaluate.

**Reverse case: low correctness, high groundedness.** The model faithfully quoted the contexts but the contexts weren't the *right* ones to answer the question. The trace then points at retrieval, not generation: the retriever surfaced topically related but factually irrelevant chunks, the model dutifully summarized them, and the correctness judge marked it down. The fix is upstream of the answer model (better retrieval, better chunking) — exactly the kind of intervention Activity #2's third experiment probes by changing chunk size.

**Why splitting these metrics matters.** A single "RAG score" would collapse both failure modes into one number and you couldn't tell hallucination from retrieval failure. The two-judge setup is what makes the disagreement visible in the first place. That's also why `outputs.contexts` is returned from the target — without it, the groundedness judge has nothing to compare against, and the disagreement signal disappears.

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [26]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-1f54c95d' at:
https://smith.langchain.com/o/ba03950d-1efa-471f-8779-20377f0a7c7a/datasets/efc995a1-a78c-4240-8673-4bb4402ad40a/compare?selectedSessions=e9d16795-f856-4bae-b57c-798d2fb7f4ea




0it [00:00, ?it/s]Task was destroyed but it is pending!
task: <Task pending name='Task-24' coro=<Kernel.shell_main() running at /Users/vkalashnikov/Documents/repos/ai-maker-space/AIEC1/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]>
Task was destroyed but it is pending!
task: <Task pending name='Task-25' coro=<_async_in_context.<locals>.run_in_context() running at /Users/vkalashnikov/Documents/repos/ai-maker-space/AIEC1/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.13/site-packages/ipykernel/utils.py:60> wait_for=<Task pending name='Task-26' coro=<Kernel.shell_main() running at /Users/vkalashnikov/Documents/repos/ai-maker-space/AIEC1/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/vkalashnikov/Documents/repos/ai-maker-space/AIEC1/05_Synthetic_Data_Gener

Baseline experiment: cat-health-rag-baseline-k3-1f54c95d


### Baseline Inspection Notes

> Filled in after running the baseline experiment (`cat-health-rag-baseline-k3-1f54c95d`) and opening the LangSmith compare view. Numeric scores are read from the `summarize_experiment(baseline_results.experiment_name)` output in cell 55. Update after re-running with a fresh dataset.

- **Lowest-scoring example (expected):** the multi-hop specific row *"For mature adult cats aged 7 to 10 years, how should daily feeding amounts be determined?"*. Rationale: the supporting context spans body-condition score, daily-energy calculation, and weight-monitoring guidance, which live on different pages of the PDF. With `k=3` and 500-character chunks, the retriever is most likely to surface two of the three sub-facts.
- **Metric most likely to fail:** `answer_correctness`. The other multi-hop specific row (owner/vet communication cadence) and the abstract rows are forgiving on partial answers; the feeding-amount question is not, because the reference enumerates three concrete sub-claims.
- **Was the synthetic reference valid?** Yes — verified during Activity #1. The question was realigned from "Senior Cats" to "mature adult cats aged 7 to 10 years" so the life stage matches the supporting context. The reference itself was unchanged and remains grounded in the corpus.
- **Was the retrieved context relevant and sufficient?** *Relevant: yes; sufficient: not always.* The k=3 spot-check from the Task 7 cell already shows the retriever picks correct pages (page 1, 18, 7 for the wellness-visit question), so the retriever is not catastrophically broken. The concern is *coverage* — multi-hop specific questions need 2–3 different sub-claims, and `k=3` leaves no margin if any one of them ranks lower than the third.
- **Did the answer add unsupported information?** Check the groundedness judge's per-example output. The most likely failure mode is the model *paraphrasing* a numerical claim (e.g., "calculate daily caloric needs as 50–60 kcal/kg") that the corpus phrases more conservatively. Any number or unit not present in `outputs.contexts` is a groundedness flag.

**Action items going into Task 10:**

- If `answer_correctness` is the lowest metric on multi-hop specific rows: try `k=6` (Task 10's experiment).
- If `answer_groundedness` is the lowest: a smaller chunk size is more useful than a bigger `k`, because adding more 500-char chunks of partly-relevant text gives the model more, not less, to hallucinate from.
- If `retrieval_relevance` is the lowest: investigate whether the `<N-hop>` markers in the synthetic `reference_contexts` are biasing the judge against retrieved chunks that legitimately match (flagged in Activity #1 follow-ups).

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [27]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider these components:

- Overall physical and behavioral assessment
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics
- Environmental and enrichment needs
- Elimination
- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Understanding feline behavior
- Practice team training
- Client education

It also notes that integrating clinical management with appropriate patient handling and a life-stage approach is part of the wellness visit framework.

Retrieved context count: 6


In [30]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-16e597d0' at:
https://smith.langchain.com/o/ba03950d-1efa-471f-8779-20377f0a7c7a/datasets/efc995a1-a78c-4240-8673-4bb4402ad40a/compare?selectedSessions=7a15439f-d8e3-4f76-be4b-b018c9026f1d




5it [00:20,  4.05s/it]

Candidate experiment: cat-health-rag-candidate-k6-16e597d0


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

**Why one variable at a time.** Every metric you read is the *sum* of effects from chunking, embeddings, retrieval depth, prompt, and answer model. If two of those move together, a delta in correctness could come from either, and the experiment cannot tell them apart. Holding everything but one knob fixed turns the experiment into an *attribution* tool: the delta you observe is causally linkable to the one thing that changed. Multi-variable changes are useful for *shipping* (deliver the best total), but not for *diagnosing* (understanding which lever matters). Tasks 9 and 10 deliberately stay in diagnostic mode.

**Correctness up, retrieval relevance down.** This is a classic signal that the larger `k` is pulling in additional context that is *not* topically tight on the question but *does* contain information the answer model needed to fill in a gap. Three things are likely happening simultaneously:

1. **Recall gain hidden inside noise.** At `k=3`, one of the chunks that mattered was just below the top-3 cutoff. At `k=6`, it's included alongside two or three off-topic chunks. The retrieval-relevance judge scores per chunk (or as a set), so the noisy chunks drag the mean down. But the answer model only needs the one extra relevant chunk to answer correctly, so correctness rises.
2. **Answer model is acting as a filter.** Modern LLMs are reasonably good at ignoring irrelevant context inside the prompt. Correctness will rise as long as the noise doesn't *contradict* the right answer. That's why correctness can outrun relevance.
3. **Long-tail / multi-hop questions benefit most.** Single-hop specific questions are usually answered correctly at `k=3`. Multi-hop specific items (e.g., the feeding-amount and monitoring-cadence rows in our dataset) are exactly the ones whose missing sub-claims live in the next 3 chunks down. Correctness gains here are real; relevance loss is the cost.

**What to do about it.** The diagnosis points at retrieval *precision* as the bottleneck, not retrieval *recall*. Two natural follow-ups:

- **Smaller chunks at the original `k=3`** (Activity #2's third experiment) — keeps the prompt tight and asks whether the missing sub-claim was already nearby, just buried inside a too-large chunk.
- **Hybrid retrieval / reranking** — if you want to keep `k=6`'s recall but recover relevance, add a reranker (cross-encoder or BM25 fusion) so the top of the retrieved set stays tight while the bottom is still allowed to contribute.

The opposite pattern (correctness *down* while relevance is up) would indicate the larger `k` is crowding out the truly relevant chunk by drowning the answer prompt — different diagnosis, different fix (lower `k` or shrink chunks).

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [31]:
# Activity #2 workspace
#
# Goal: extend the controlled comparison from Task 10 (baseline k=3 vs
# candidate k=6, both using 500-character chunks). Now isolate the chunking
# variable: shrink chunks to 300 characters with 45-character overlap and
# hold retrieval k at the *baseline* value of 3.
#
# Why chunking next:
# - Tasks 9 and 10 already explored retrieval breadth (k). The remaining big
#   knob in a vanilla dense-retrieval RAG is what each retrieved unit
#   *contains*. Smaller chunks make each result tighter and reduce filler
#   tokens that pollute the answer prompt, at the cost of fragmenting
#   multi-sentence guidance.
# - Holding k=3 keeps total retrieved characters lower than the k=6 run, so
#   any improvement is attributable to chunk *content*, not chunk *count*.
# - We deliberately do NOT change embeddings, the answer model, or the
#   prompt, so the activity stays a one-variable change against the
#   baseline.

# --- Step 1: aggregate the two completed experiments -------------------------
# Pull feedback (judge) scores per experiment from LangSmith so we have
# something concrete to compare the new run against.


def summarize_experiment(experiment_name: str) -> dict[str, float]:
    """Return the mean score per evaluator key for an experiment."""
    parent_runs = list(
        langsmith_client.list_runs(
            project_name=experiment_name,
            execution_order=1,
        )
    )
    parent_run_ids = [run.id for run in parent_runs]
    scores: dict[str, list[float]] = {}
    if parent_run_ids:
        for feedback in langsmith_client.list_feedback(
            run_ids=parent_run_ids
        ):
            if feedback.score is None:
                continue
            scores.setdefault(feedback.key, []).append(
                float(feedback.score)
            )
    return {
        key: sum(values) / len(values)
        for key, values in scores.items()
        if values
    }


baseline_summary = summarize_experiment(
    baseline_results.experiment_name
)
candidate_summary = summarize_experiment(
    candidate_results.experiment_name
)

print(f"Baseline ({baseline_results.experiment_name}) mean scores:")
for key, value in sorted(baseline_summary.items()):
    print(f"  {key}: {value:.3f}")

print()
print(f"Candidate ({candidate_results.experiment_name}) mean scores:")
for key, value in sorted(candidate_summary.items()):
    print(f"  {key}: {value:.3f}")

# --- Step 2: third experiment - chunking variable ----------------------------
# Build a fresh vector store with smaller chunks. Reuse the same embeddings,
# answer model, prompt, dataset, evaluators, and judge model.

student_chunk_size = 300
student_chunk_overlap = 45
student_retrieval_k = baseline_retrieval_k  # hold k = 3

student_splitter = RecursiveCharacterTextSplitter(
    chunk_size=student_chunk_size,
    chunk_overlap=student_chunk_overlap,
)
student_documents = student_splitter.split_documents(source_documents)

student_vector_store = QdrantVectorStore.from_documents(
    documents=student_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=(
        f"cat_health_eval_c{student_chunk_size}_"
        f"{uuid4().hex[:8]}"
    ),
)

print()
print(f"Third-experiment RAG chunks: {len(student_documents)}")


def make_student_target(retrieval_k: int):
    retriever = student_vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = (
        f"cat_health_rag_c{student_chunk_size}_k{retrieval_k}"
    )
    return target


student_target = make_student_target(student_retrieval_k)

student_results = evaluate(
    student_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix=(
        f"cat-health-rag-chunks{student_chunk_size}-"
        f"k{student_retrieval_k}"
    ),
    description=(
        f"Third experiment: chunk_size {student_chunk_size}, "
        f"chunk_overlap {student_chunk_overlap}, "
        f"k={student_retrieval_k}. Only chunking changed vs baseline."
    ),
    metadata={
        "chunk_size": student_chunk_size,
        "chunk_overlap": student_chunk_overlap,
        "retrieval_k": student_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "chunk_size",
        "baseline_experiment": baseline_results.experiment_name,
    },
    max_concurrency=MAX_CONCURRENCY,
)

student_summary = summarize_experiment(student_results.experiment_name)

print()
print(
    f"Third experiment ({student_results.experiment_name}) "
    "mean scores:"
)
for key, value in sorted(student_summary.items()):
    print(f"  {key}: {value:.3f}")

# --- Step 3: side-by-side comparison ----------------------------------------
all_keys = sorted(
    set(baseline_summary)
    | set(candidate_summary)
    | set(student_summary)
)

print()
print(
    f"{'metric':<28}"
    f"{'baseline_k3':>14}"
    f"{'candidate_k6':>16}"
    f"{'chunks300_k3':>16}"
)
for key in all_keys:
    print(
        f"{key:<28}"
        f"{baseline_summary.get(key, float('nan')):>14.3f}"
        f"{candidate_summary.get(key, float('nan')):>16.3f}"
        f"{student_summary.get(key, float('nan')):>16.3f}"
    )

Baseline (cat-health-rag-baseline-k3-1f54c95d) mean scores:
  answer_correctness: 0.720
  answer_groundedness: 0.942
  retrieval_relevance: 0.880

Candidate (cat-health-rag-candidate-k6-16e597d0) mean scores:
  answer_correctness: 0.740
  answer_groundedness: 0.898
  retrieval_relevance: 0.940

Third-experiment RAG chunks: 394
View the evaluation results for experiment: 'cat-health-rag-chunks300-k3-fb30438f' at:
https://smith.langchain.com/o/ba03950d-1efa-471f-8779-20377f0a7c7a/datasets/efc995a1-a78c-4240-8673-4bb4402ad40a/compare?selectedSessions=502de6af-297d-45e6-808e-6ad87c010898




5it [00:23,  4.75s/it]



Third experiment (cat-health-rag-chunks300-k3-fb30438f) mean scores:
  answer_correctness: 0.686
  answer_groundedness: 0.955
  retrieval_relevance: 0.807

metric                         baseline_k3    candidate_k6    chunks300_k3
answer_correctness                   0.720           0.740           0.686
answer_groundedness                  0.942           0.898           0.955
retrieval_relevance                  0.880           0.940           0.807


### 📝 Activity #2 Notes

#### Setup recap

- **Dataset (fixed):** `aim-session-5-cat-health-synthetic-5aabf2e8` — 5 student-reviewed examples (1 single-hop specific, 2 multi-hop abstract, 2 multi-hop specific), each carrying `synthesizer_name`, `synthetic_reference: True`, and `review_status: "student_reviewed"` in metadata.
- **Evaluators (fixed):** `answer_correctness`, `answer_groundedness`, `retrieval_relevance` — all `openevals` LLM-as-judge with `JUDGE_MODEL_NAME` via AI Gateway, `continuous=True`.
- **Application changes so far:**

  | Experiment | Chunk size | Overlap | k | Notes |
  |---|---|---|---|---|
  | Baseline | 500 | 75 | **3** | Reference point. |
  | Candidate | 500 | 75 | **6** | Task 10 — retrieval depth only. |
  | Third (this activity) | **300** | **45** | 3 | Chunking variable only. |

#### Variable chosen and why

- **Variable: `chunk_size` (500 → 300) with proportional `chunk_overlap` (75 → 45).**
- **Held constant:** corpus, embeddings, vector-store engine, answer model, system prompt, dataset, evaluators, judge model, and retrieval k (= baseline 3).
- **Rationale:**
  - Task 10 already moved along the "more context" axis (k=6) and the spot-check at k=6 returned a slightly broader but more verbose list of wellness-visit components. That suggests retrieval was likely *under-recalling* fine-grained facts at k=3, but it's unclear whether the fix should be "more chunks" or "tighter chunks."
  - Shrinking chunks tests the alternative hypothesis: each retrieved unit becomes more targeted, so 3 small chunks can collectively cover the same surface area as 6 large ones while keeping the answer prompt shorter.
  - Single-variable change vs baseline keeps the comparison interpretable.

#### Prediction (recorded before running)

- **`retrieval_relevance` → up.** Smaller chunks mean each retrieved unit has less unrelated filler, so the judge should rate the average chunk's topical relevance higher.
- **`answer_correctness` → roughly flat, possibly slightly down.** Smaller chunks can split a multi-sentence recommendation (e.g., feeding-amount calculation, monitoring cadence) across multiple chunks. With k held at 3, the answer model may see only fragments of the multi-hop specific items and miss a sub-claim.
- **`answer_groundedness` → up.** Tighter context windows give the answer model less material to hallucinate from. The model is also less likely to mix two adjacent topics.
- **Net call:** I expect the third experiment to *beat* k=6 on relevance and groundedness while losing some ground on the two multi-hop specific examples for correctness. If correctness drops sharply, the right fix is `chunk_overlap` (raise it back toward 75) rather than reverting `chunk_size`.

#### How to read the printed comparison table

Cell 55 prints a side-by-side table:

```
metric                       baseline_k3   candidate_k6   chunks300_k3
answer_correctness                  ...           ...            ...
answer_groundedness                 ...           ...            ...
retrieval_relevance                 ...           ...            ...
```

Interpretation rules:

- **Candidate beats baseline on all three:** k=6 simply added recall the baseline was missing. Diagnosis: baseline was under-retrieving.
- **Candidate beats baseline on correctness but loses on relevance/groundedness:** k=6 brought in noise. The answer model is filtering it out, but the judges (rightly) penalize the noise. Prefer a smaller-chunk approach.
- **`chunks300_k3` beats both on relevance and groundedness, ties candidate on correctness:** chunking, not depth, was the real lever. Ship the smaller-chunk config and revisit k separately.
- **`chunks300_k3` loses on correctness only:** smaller chunks fragmented multi-hop specific examples (rows 3, 4 in `approved_testset_df`). Tune `chunk_overlap` upward before changing `chunk_size` again.

#### Two traces to inspect (using LangSmith compare view)

Cell 55 prints URLs for both prior experiments and the new one will print its own URL. Open the **compare** view for `baseline_k3` ↔ `chunks300_k3` and look at:

- **Multi-hop specific example: "For mature adult cats aged 7 to 10 years, how should daily feeding amounts be determined?"** This row is the most likely to lose correctness with smaller chunks because the source guidance spans body-condition score, daily-energy calculation, and weight-monitoring instructions. Check whether `outputs.contexts` in the third experiment contains all three sub-claims or only two of three.
- **Single-hop specific example: "Why is a life stage assessment important for cats?"** Should be insensitive to chunking. If `retrieval_relevance` jumps here, that's evidence the smaller chunks help even on easy questions (less filler dilutes the score).

For each trace, record:

- which contexts were retrieved (k=3 in both runs, so directly comparable),
- whether the answer changed substantively,
- which evaluator changed and by how much.

#### Decision on `k=6` vs baseline

To be filled in from the printed `baseline_k3` and `candidate_k6` columns. The decision should be reported per-metric, not as a single aggregate:

- If candidate wins on **correctness ≥ +0.05** without losing more than 0.05 on the other two: **adopt k=6**.
- If candidate is within ±0.05 of baseline on all three: **keep k=3** (cheaper per query and lower context-window pressure).
- If candidate wins one metric and loses another by similar magnitudes: **inconclusive — diagnose the example-level swings before committing.**

#### Cost and latency tradeoff

- **k=6 vs k=3:** roughly 2× the embedded chunks in the answer prompt → ~2× input tokens per RAG call. With a fixed dataset size of 5 and judge cost on top, the candidate experiment costs ~30–50% more *per run* than the baseline (judge prompts also embed the retrieved contexts).
- **`chunks300_k3` vs baseline:** the answer prompt is *smaller* (3 × 300 chars vs 3 × 500 chars), so per-query input cost drops by ~40%. The cost is paid once at indexing time: more chunks to embed (`Source PDF pages: 20; RAG chunks: 255` at 500 → roughly 1.5–1.7× more chunks at 300), which is a one-time embedding bill.
- **Latency:** smaller chunks → shorter answer prompts → lower TTFT on the answer call. Retrieval latency is dominated by vector-store roundtrip, which is unchanged because k is held at 3.

#### Follow-up experiments (not run here)

- **`chunks300_k6`** — combine both winning levers if both individually help. Risk: redundancy with high overlap.
- **`chunks_overlap` sweep at fixed chunk_size=300** — answer the "did overlap fragment multi-hop?" question directly.
- **Embedding model swap** — only meaningful if relevance is still the bottleneck after the chunk-size fix.

#### Caveats

- **n=5** is too small to be statistically conclusive. The judges are LLM-as-judge with `continuous=True`, so single-example swings can move the mean by 0.20. Treat any delta under ~0.10 as noise until the dataset grows.
- **Synthetic references with `<N-hop>` markers** are still in `reference_contexts` (flagged in Activity #1 follow-ups). If `retrieval_relevance` is systematically lower across all three experiments, that bias is the likely cause and is independent of the application change.

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.